# kaNAIRO — free Kaggle image-to-3D asset lab

Project rules: read `skills/kaggle-asset-lab/SKILL.md`, `skills/blender-nairobi-game-art/SKILL.md`, and `docs/BLENDER_ASSETS.md` in the cloned game repository before preparing a model for the game.

TripoSR reconstructs a 3D guess from an image; it does not make assets from text, promise correct unseen sides, create game animation, or guarantee production-ready topology. Start with one clean reference and review the result in Blender before integrating it. Only upload images you have the right to use.



In [ ]:
from pathlib import Path
import subprocess, sys, json, os

assert Path('/kaggle').exists(), 'Run this notebook on Kaggle.'
subprocess.run(['nvidia-smi'], check=False)
WORK = Path('/kaggle/working/nairoblife-assets')
WORK.mkdir(parents=True, exist_ok=True)
print('Outputs will be written to:', WORK)

## Clone the project instructions and TripoSR

The game clone is only for its project-specific skill, asset rules and provenance notes. TripoSR code and pretrained checkpoint are MIT-licensed according to its upstream README. Check the upstream license and current dependency instructions before each new run.

In [ ]:
GAME = Path('/kaggle/working/NairobLife')
TRIPOSR = Path('/kaggle/working/TripoSR')
if not GAME.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'arena/f8aec938-nairoblife', '--single-branch', 'https://github.com/wambetebenjamin/NairobLife.git', str(GAME)], check=True)
if not TRIPOSR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/VAST-AI-Research/TripoSR.git', str(TRIPOSR)], check=True)
print((GAME / 'skills/kaggle-asset-lab/SKILL.md').read_text())
print((GAME / 'skills/blender-nairobi-game-art/SKILL.md').read_text()[:5000])
print('TripoSR checkout:', subprocess.check_output(['git','-C',str(TRIPOSR),'rev-parse','HEAD'], text=True).strip())

## Install TripoSR dependencies

Kaggle already provides PyTorch in many GPU images. The upstream requirements file intentionally does not pin/install PyTorch. Dependency resolution or CUDA extension compilation can still fail when Kaggle's image changes; stop and inspect the error rather than blindly upgrading the preinstalled CUDA/PyTorch stack.

In [ ]:
%cd /kaggle/working/TripoSR
!python -m pip install -q --upgrade setuptools
!python -m pip install -q -r requirements.txt
!python -m pip install -q git+https://github.com/tatsy/torchmcubes.git
import torch
print('torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before running model inference.')

## Find rights-cleared Nairobi reference media

With Kaggle internet enabled, this searches **Wikimedia Commons** (not Google scraping or YouTube downloads) for Nairobi wildlife, urban scenes, and short video references. It downloads only files with identifiable CC0, public-domain, CC BY, or CC BY-SA terms, and records author/license/source metadata. CC BY requires attribution; CC BY-SA also carries ShareAlike obligations, so review its terms before using it in a derived asset. This does not auto-select a model input: inspect the downloaded references first.

In [ ]:
import hashlib, html, mimetypes, re, shutil, time
from datetime import datetime, timezone
from urllib.parse import quote
import requests

MEDIA_DIR = WORK / 'commons-media'
MEDIA_DIR.mkdir(parents=True, exist_ok=True)
COMMONS_API = 'https://commons.wikimedia.org/w/api.php'
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'kaNAIRO-KaggleAssetLab/1.0 (public educational game asset research)'})

def plain(value):
    if isinstance(value, dict): value = value.get('value', '')
    return html.unescape(re.sub(r'<[^>]+>', '', str(value or ''))).strip()

def reusable_license(label):
    text = label.lower().replace('creativecommons', 'cc')
    if 'noncommercial' in text or 'non-commercial' in text or 'no derivatives' in text or 'no-derivatives' in text or 'cc by-nc' in text or 'cc by-nd' in text:
        return False
    return ('cc0' in text or 'public domain' in text or 'cc by' in text or 'attribution' in text)

def search_commons(query, kind, max_results=2):
    filetype = 'video' if kind == 'video' else 'bitmap'
    params = {'action':'query','format':'json','generator':'search','gsrnamespace':6,'gsrlimit':30,'gsrsearch':f'{query} filetype:{filetype}','prop':'imageinfo','iiprop':'url|mime|size|extmetadata','iiurlwidth':1400}
    response = SESSION.get(COMMONS_API, params=params, timeout=60)
    response.raise_for_status()
    pages = response.json().get('query', {}).get('pages', {})
    found = []
    for page in sorted(pages.values(), key=lambda p: p.get('index', 9999)):
        info_list = page.get('imageinfo') or []
        if not info_list: continue
        info = info_list[0]
        mime = info.get('mime', '')
        if (kind == 'video' and not mime.startswith('video/')) or (kind == 'image' and not mime.startswith('image/')): continue
        ext = info.get('extmetadata', {})
        license_name = plain(ext.get('LicenseShortName')) or plain(ext.get('UsageTerms'))
        if not reusable_license(license_name): continue
        if kind == 'video' and info.get('size', 0) > 120 * 1024 * 1024: continue
        url = info.get('url') if kind == 'video' else info.get('thumburl', info.get('url'))
        found.append({'page':page,'info':info,'mime':mime,'license':license_name,'url':url,'ext':ext})
        if len(found) >= max_results: break
    return found

topics = [
    ('Nairobi National Park giraffe','image'),
    ('Nairobi National Park zebra','image'),
    ('Nairobi National Park rhinoceros','image'),
    ('Nairobi National Park wildlife','image'),
    ('Nairobi acacia Kenya','image'),
    ('Nairobi skyline Kenya','image'),
    ('Nairobi matatu street','image'),
    ('Nairobi National Park giraffe','video'),
    ('Nairobi National Park zebra','video'),
    ('Nairobi Kenya street','video'),
    ]
media_manifest = []
seen_pages = set()
for query, kind in topics:
    try:
        candidates = search_commons(query, kind, max_results=1 if kind == 'video' else 2)
    except Exception as exc:
        print('Search failed:', query, type(exc).__name__, str(exc)[:240])
        continue
    for item in candidates:
        page = item['page']; info = item['info']
        if page['pageid'] in seen_pages: continue
        seen_pages.add(page['pageid'])
        suffix = mimetypes.guess_extension(item['mime']) or ('.webm' if kind == 'video' else '.jpg')
        if suffix == '.jpe': suffix = '.jpg'
        dest = MEDIA_DIR / f"{kind}-{len(media_manifest)+1:02d}{suffix}"
        try:
            with SESSION.get(item['url'], stream=True, timeout=(30,180)) as r:
                r.raise_for_status()
                written = 0
                with dest.open('wb') as f:
                    for chunk in r.iter_content(1024*1024):
                        if not chunk: continue
                        written += len(chunk)
                        if written > (120 if kind == 'video' else 20) * 1024 * 1024: raise ValueError('download exceeds configured size cap')
                        f.write(chunk)
            record = {'kind':kind,'query':query,'title':page.get('title'),'page_url':'https://commons.wikimedia.org/wiki/'+quote(page.get('title','').replace(' ','_')),'source_url':info.get('descriptionurl'),'download_url':item['url'],'mime':item['mime'],'license':item['license'],'artist':plain(item['ext'].get('Artist')),'credit':plain(item['ext'].get('Credit')),'usage_terms':plain(item['ext'].get('UsageTerms')),'description':plain(item['ext'].get('ImageDescription'))[:1200],'file':str(dest),'bytes':dest.stat().st_size,'sha256':hashlib.sha256(dest.read_bytes()).hexdigest(),'downloaded_utc':datetime.now(timezone.utc).isoformat()}
            media_manifest.append(record)
            print(kind, record['license'], record['title'], '->', dest.name, f"({record['bytes']:,} bytes)")
        except Exception as exc:
            dest.unlink(missing_ok=True)
            print('Download skipped:', page.get('title'), type(exc).__name__, str(exc)[:200])
        time.sleep(0.15)

(MEDIA_DIR / 'media-manifest.json').write_text(json.dumps(media_manifest, indent=2, ensure_ascii=False))
print(f'Fetched {len(media_manifest)} licensed reference files into {MEDIA_DIR}')
print('Review titles and licenses; do not assume each search result depicts the query exactly.')

## Extract a few review frames from downloaded videos

Frames are visual references only. TripoSR is single-image conditioned, so choose one clean frame as its input; do not pass a video or unrelated frames as though they were multi-view reconstruction.

In [ ]:
import shutil, subprocess
FRAME_DIR = WORK / 'video-reference-frames'
FRAME_DIR.mkdir(parents=True, exist_ok=True)
frame_records = []
ffmpeg = shutil.which('ffmpeg')
if not ffmpeg:
    print('ffmpeg is not available in this Kaggle image; keep the downloaded videos as references or install a compatible ffmpeg package.')
else:
    for record in media_manifest:
        if record['kind'] != 'video': continue
        target = FRAME_DIR / (Path(record['file']).stem + '-%02d.jpg')
        cmd = [ffmpeg, '-hide_banner', '-loglevel', 'error', '-i', record['file'], '-vf', 'fps=1/8,scale=1280:-2', '-frames:v', '8', '-y', str(target)]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print('Frame extraction failed:', Path(record['file']).name, result.stderr[-400:])
            continue
        for frame in sorted(FRAME_DIR.glob(Path(record['file']).stem + '-*.jpg')):
            frame_records.append({'file':str(frame),'video_title':record['title'],'video_page_url':record['page_url'],'license':record['license'],'artist':record['artist'],'sha256':hashlib.sha256(frame.read_bytes()).hexdigest()})
(FRAME_DIR / 'frames-manifest.json').write_text(json.dumps(frame_records, indent=2, ensure_ascii=False))
print('Extracted reference frames:', len(frame_records), 'directory:', FRAME_DIR)

## Select one reference image

Choose a reviewed image from `/kaggle/input`, the downloaded Commons image folder, or the extracted video frames listed by the cell below, then paste that exact path into `IMAGE`. Prefer a single, isolated object on a plain background; whole scenes and multiple objects produce poor game meshes. For a car, use a clear three-quarter view; for a standing person, use a full-body view with limbs visible. The model still infers unseen sides from that one view.

In [ ]:
from pathlib import Path
from PIL import Image

inputs = list(Path('/kaggle/input').rglob('*')) if Path('/kaggle/input').exists() else []
web_refs = list(MEDIA_DIR.rglob('*')) + list(FRAME_DIR.rglob('*'))
images = [p for p in inputs + web_refs if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.webp'}]
print('Kaggle inputs, Commons images, and extracted video reference frames:')
for p in images[:40]: print(p)
# CHANGE this to one image listed above, from a source you are allowed to use.
IMAGE = Path('/kaggle/input/your-reference-dataset/your-reference.png')
if not IMAGE.is_file():
    raise FileNotFoundError('Set IMAGE to a real image path under /kaggle/input, then rerun this cell.')
display(Image.open(IMAGE).convert('RGB').resize((512,512)))

## Generate one GLB candidate

This asks TripoSR for a 1K baked texture and binary glTF output. If texture baking runs out of memory, retry once without `--bake-texture`; the default output uses vertex colors. Do not bulk-generate until the first model has been inspected.

In [ ]:
import datetime, shlex
ASSET_NAME = 'candidate'  # e.g. 'nairobi-hatchback' or 'pedestrian-rosa'
OUT = WORK / ASSET_NAME
OUT.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, 'run.py', str(IMAGE), '--device', 'cuda:0', '--model-save-format', 'glb', '--bake-texture', '--texture-resolution', '1024', '--output-dir', str(OUT)]
print('Running:', shlex.join(cmd))
subprocess.run(cmd, cwd=TRIPOSR, check=True)
glbs = sorted(OUT.rglob('*.glb'))
print('GLB outputs:', glbs)
assert glbs, 'No GLB output found; inspect the inference log and TripoSR docs.'
manifest = {'asset_name': ASSET_NAME, 'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'source_image_name': IMAGE.name, 'source_repo': 'https://github.com/VAST-AI-Research/TripoSR', 'source_commit': subprocess.check_output(['git','-C',str(TRIPOSR),'rev-parse','HEAD'], text=True).strip(), 'model': 'stabilityai/TripoSR', 'license_note': 'TripoSR README states code and pretrained model are MIT; verify current terms before distribution.', 'review_status': 'candidate only; inspect orientation, scale, topology, materials, rights and game performance before use'}
(OUT / 'asset-manifest.json').write_text(json.dumps(manifest, indent=2))
for p in glbs: print(p, p.stat().st_size, 'bytes')

## Review and bring back to the game

Kaggle output is not game-ready simply because it is GLB. Inspect in Blender: check dimensions in metres, ground contact/origin, Y-up and forward -Z, normals, topology, texture, transparency and visual quality. The game skill targets under 2 MB per hero GLB where practical and requires a Three.js fallback. TripoSR does not rig people or create animation clips, so do not use its output to replace the animated player/crowd without a separate rigging/animation pass.

Download only the reviewed `.glb` and `asset-manifest.json` from Kaggle outputs. Add it to `public/assets/` only after it has been integrated and tested in the actual game; update `docs/FREE_ASSETS.md` with provenance. Never copy Kaggle checkpoints, caches, arbitrary input photos, or tokens into Git.

## Create clickable download links

After inference finishes, run the cell below. It bundles all files under the asset workspace—including GLBs, downloaded Commons images/videos, extracted video frames, and provenance manifests—into one ZIP and renders one clickable download link. In the current Kaggle session, you can also paste/run this cell even if you imported an older notebook version.

In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import zipfile
WORK = Path('/kaggle/working/nairoblife-assets')
bundle = WORK / 'candidate-download.zip'
files = sorted(p for p in WORK.rglob('*') if p.is_file() and p != bundle)
if not files:
    print('No files found. Run the Commons download and/or model generation cells first.')
else:
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        for p in files:
            z.write(p, p.relative_to(WORK))
    print(f'Bundled {len(files)} files ({bundle.stat().st_size:,} bytes): images, videos, extracted frames, models, and manifests.')
    display(FileLink(str(bundle)))
